# 07 — Penalaan & Peningkatan Model pada Data VPS Perusahaan

**Tujuan:** menurunkan alarm palsu dan meningkatkan deteksi model A+C pada lalu lintas server VPS PT Hotdog Water
Sport, dengan prosedur yang adil: pengaturan dipilih pada **data validasi**, lalu diuji **satu kali** pada **data uji**
yang tidak pernah dipakai untuk memilih.

**Data:** seluruh rekaman header 22 Okt – 11 Nov 2026 (±20 hari), dibagi menurut waktu:
latih 22 Okt – 3 Nov · validasi 4 – 7 Nov · uji 8 – 11 Nov (UTC).

**Yang dicoba** (temuan dari pemeriksaan alarm pada pengunjung):
- **V0 – rancangan awal:** standarisasi saja, pembersihan sumber intens per IP.
- **V1 – + pembersihan subnet:** subnet (/24 IPv4, /64 IPv6) yang dalam satu jendela menghubungi ≥ 20 port berbeda
  atau terdiri dari ≥ 20 IP berbeda (*pemindaian terdistribusi*) ikut dikeluarkan dari data latih.
- **V2 – + transformasi log:** setiap fitur diubah menjadi sign(x)·log(1+|x|) sebelum distandarkan. Fitur jaringan
  sangat menceng (mis. byte sesi unduhan besar), sehingga tanpa transformasi nilai ekstrem yang sah mudah terisolasi.
- **V3 – V1 + V2.**

Masing-masing dicoba dengan beberapa nilai *contamination* A dan C.

**Aturan pemilihan:** IP penyerang yang terdeteksi (serangan intens + pemindaian terdistribusi + pemindai lambat)
**sebanyak mungkin**, dengan syarat **alarm palsu pada pengunjung ≤ 1% aliran** (rancangan awal: 1,9%).

In [ ]:
import os, sys, glob, json, time
import numpy as np
import pandas as pd

FOLDER = os.path.abspath('')                                              # 04_Kode/4_notebook
KODE = os.path.abspath(os.path.join(FOLDER, '..', '2_sistem_nids'))
DATA = os.path.abspath(os.path.join(FOLDER, '..', '..', '05_Data_Eksperimen', 'Server-HotDog'))
CIC = os.path.abspath(os.path.join(FOLDER, '..', '..', '05_Data_Eksperimen', 'CIC-IDS2017'))
sys.path.insert(0, KODE)
from nids.pipa import putar_ulang, KOLOM
from nids.model import latih, SatuModel
from nids.agregasi import FITUR_AGREGASI, subnet
from nids.aliran import FITUR
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 250)

SERVER_IP = {'192.0.2.20', '2a02:4780:5e:4c51::1'}
B_VAL, B_UJI = pd.Timestamp('2026-11-04', tz='UTC'), pd.Timestamp('2026-11-08', tz='UTC')
BATAS_PENGUNJUNG = 1.0                    # % aliran pengunjung yang boleh ikut ditandai (syarat pemilihan)
HASIL = os.path.join(DATA, 'hasil'); os.makedirs(HASIL, exist_ok=True)

## 1. Preprocessing: seluruh rekaman → aliran berfitur (pipa yang sama dengan sistem NIDS)

In [ ]:
berkas = sorted(glob.glob(os.path.join(DATA, 'rekaman', '*.pcap.gz')))
t0 = time.time()
aliran, pipa = putar_ulang(berkas, 300)
df = pd.DataFrame(aliran)
df = df[[c for c in KOLOM + ['_mulai_us'] if c in df.columns]]
df.to_parquet(os.path.join(HASIL, 'fitur_vps_perusahaan.parquet'), index=False)
df['Waktu'] = pd.to_datetime(df['Waktu Selesai'], unit='us', utc=True)
df['Masuk'] = ~df['Source IP'].isin(SERVER_IP)
df['Subnet'] = df['Source IP'].map(subnet)
print('%d berkas | %s paket | %s aliran | %.0f detik' % (len(berkas), f'{pipa.jumlah_paket:,}', f'{len(df):,}', time.time() - t0))
print('Rentang (UTC): %s s.d. %s' % (df.Waktu.min().strftime('%d-%m-%Y %H:%M'), df.Waktu.max().strftime('%d-%m-%Y %H:%M')))

## 2. Pembagian waktu & label berbasis aturan

Label hanya untuk **memeriksa** hasil (Isolation Forest tidak memakai label saat dilatih). Dibanding notebook 03,
ada kategori baru **pemindaian terdistribusi**: banyak alamat dari satu subnet yang masing-masing hanya mengetuk
sekali, tetapi bersama-sama menghubungi puluhan sampai ratusan port. Sebelumnya IP-IP ini keliru terhitung sebagai
"pengunjung" sehingga alarmnya tampak seperti alarm palsu.

| Kategori | Aturan (dihitung per periode) |
|---|---|
| Serangan intens | IP pernah ≥ 20 port berbeda atau ≥ 20 koneksi SSH dalam 5 menit |
| Pemindaian terdistribusi | subnet IP itu pernah ≥ 20 port berbeda atau ≥ 20 IP berbeda dalam 5 menit |
| Pemindai lambat | ≥ 10 port berbeda selama periode, tanpa pernah menerima data balasan |
| Pengunjung berdata | pernah bertukar data dengan server dan bukan kategori di atas (perkiraan **normal**) |
| Lainnya | sisanya, kebanyakan bot yang mengetuk 1–2 port |

In [ ]:
def kategori(g):
    mm = g[g.Masuk]
    intens = set(mm.loc[(mm.agg_ip_port_unik >= 20) | (mm.agg_ip_aliran_ssh >= 20), 'Source IP'])
    sub_scan = set(mm.loc[(mm.agg_s24_port_unik >= 20) | (mm.agg_s24_ip_unik >= 20), 'Subnet'])
    per_ip = mm.groupby('Source IP').agg(port=('Destination Port', 'nunique'), sub=('Subnet', 'first'),
                                         data=('Total Length of Bwd Packets', lambda s: (s > 0).any()))
    k = pd.Series('Lainnya', index=per_ip.index)
    k[per_ip.data] = 'Pengunjung berdata'
    k[(per_ip.port >= 10) & ~per_ip.data] = 'Pemindai lambat'
    k[per_ip['sub'].isin(sub_scan)] = 'Pemindaian terdistribusi'
    k[k.index.isin(intens)] = 'Serangan intens'
    return g['Source IP'].map(k).fillna('Server (keluar)')

latih_df = df[df.Waktu < B_VAL].copy()
val_df = df[(df.Waktu >= B_VAL) & (df.Waktu < B_UJI)].copy()
uji_df = df[df.Waktu >= B_UJI].copy()
for g in (latih_df, val_df, uji_df):
    g['Kategori'] = kategori(g)
pd.DataFrame({n + ': IP': g[g.Masuk].groupby('Kategori')['Source IP'].nunique()
              for n, g in (('latih', latih_df), ('validasi', val_df), ('uji', uji_df))}).fillna(0).astype(int)

## 3. Varian model dan kisi *contamination* (dinilai pada data VALIDASI)

Yang dinilai hanya koneksi yang dimulai dari luar (sama seperti sistem NIDS). *Alarm/hari* = jumlah notifikasi
Telegram per hari menurut aturan sistem (≥ 3 anomali per IP dalam 5 menit atau kedua model sepakat, jeda 10 menit per IP).

In [ ]:
KAT = ['Serangan intens', 'Pemindaian terdistribusi', 'Pemindai lambat', 'Pengunjung berdata', 'Lainnya']

def alarm_per_hari(g, t, ac):
    a = g[t].assign(ac=ac[t]).sort_values('Waktu Selesai')
    terakhir, tunggu, n = {}, {}, 0
    for ip, tt, x in zip(a['Source IP'], a['Waktu Selesai'] / 1e6, a.ac):
        if tt - terakhir.get(ip, -1e18) < 600:
            continue
        w = [y for y in tunggu.get(ip, []) if tt - y <= 300] + [tt]
        if len(w) >= 3 or x:
            terakhir[ip], tunggu[ip], n = tt, [], n + 1
        else:
            tunggu[ip] = w
    return n / ((g['Waktu Selesai'].max() - g['Waktu Selesai'].min()) / 86400e6)

def latih_varian(data, buang, transformasi):
    b = data[~data.Kategori.isin(buang)]
    p = latih(b, 0.001, 0.01, 300, transformasi=transformasi)
    mA, mC = SatuModel(p['A']), SatuModel(p['C'])
    return {'mA': mA, 'mC': mC, 'sA': mA.skor(b), 'sC': mC.skor(b), 'n_latih': len(b), 'paket': p}

def nilai(v, g, cA, cC):
    ga = g[g.Masuk]
    fa = v['mA'].skor(ga) > np.quantile(v['sA'], 1 - cA)
    fc = v['mC'].skor(ga) > np.quantile(v['sC'], 1 - cC)
    t = pd.Series(fa | fc, index=ga.index)
    r, ip_serangan = {}, 0
    for k in KAT:
        s = ga.Kategori == k
        n_ip = ga.loc[s & t, 'Source IP'].nunique()
        r[k + ' (IP)'] = '%d/%d' % (n_ip, ga.loc[s, 'Source IP'].nunique())
        r[k + ' (%)'] = round(100 * t[s].mean(), 2) if s.any() else 0.0
        if k in KAT[:3]:
            ip_serangan += n_ip
    r['IP serangan terdeteksi'] = ip_serangan
    r['alarm/hari'] = round(alarm_per_hari(ga, t, pd.Series(fa & fc, index=ga.index)), 1)
    return r

VARIAN = {'V0 rancangan awal': (['Serangan intens'], None),
          'V1 + bersihkan subnet': (['Serangan intens', 'Pemindaian terdistribusi'], None),
          'V2 + log': (['Serangan intens'], 'log'),
          'V3 subnet + log': (['Serangan intens', 'Pemindaian terdistribusi'], 'log')}
model_v = {n: latih_varian(latih_df, *cfg) for n, cfg in VARIAN.items()}
kisi = []
for n, v in model_v.items():
    for cA in (0.0005, 0.001):
        for cC in (0.005, 0.0075, 0.01, 0.015, 0.02):
            kisi.append({'varian': n, 'cA': cA, 'cC': cC, **nilai(v, val_df, cA, cC)})
kisi = pd.DataFrame(kisi)
kisi.to_csv(os.path.join(HASIL, 'penalaan_validasi.csv'), index=False)
kisi[['varian', 'cA', 'cC'] + [k + ' (IP)' for k in KAT[:3]] + ['Pengunjung berdata (IP)', 'Pengunjung berdata (%)',
      'Lainnya (%)', 'IP serangan terdeteksi', 'alarm/hari']]

## 4. Pemilihan

In [ ]:
lolos = kisi[kisi['Pengunjung berdata (%)'] <= BATAS_PENGUNJUNG]
pilih = lolos.sort_values(['IP serangan terdeteksi', 'Pengunjung berdata (%)'], ascending=[False, True]).iloc[0]
V_PILIH, CA, CC = pilih.varian, float(pilih.cA), float(pilih.cC)
print('Terpilih: %s | contamination A = %g, C = %g' % (V_PILIH, CA, CC))
pilih.to_frame('validasi')

## 5. Uji SEKALI pada data uji (8 – 11 Nov): rancangan awal vs terpilih

In [ ]:
uji = pd.DataFrame({
    'V0 rancangan awal (A 0,001; C 0,01)': nilai(model_v['V0 rancangan awal'], uji_df, 0.001, 0.01),
    'Terpilih: %s (A %g; C %g)' % (V_PILIH, CA, CC): nilai(model_v[V_PILIH], uji_df, CA, CC)})
uji.to_csv(os.path.join(HASIL, 'penalaan_uji.csv'))
uji

In [ ]:
print('Alarm pada pengunjung & "lainnya" di data uji (model terpilih), untuk diperiksa satu per satu:')
v = model_v[V_PILIH]
ga = uji_df[uji_df.Masuk].copy()
ga['A'] = v['mA'].skor(ga) > np.quantile(v['sA'], 1 - CA)
ga['C'] = v['mC'].skor(ga) > np.quantile(v['sC'], 1 - CC)
fp = ga[(ga.A | ga.C) & ga.Kategori.isin(['Pengunjung berdata', 'Lainnya'])]
(fp.groupby(['Kategori', 'Source IP']).agg(aliran=('Flow ID', 'size'), model=('A', lambda s: 'A' if s.all() else ('C' if not s.any() else 'A/C')),
                                           port=('Destination Port', lambda s: ', '.join(map(str, s.value_counts().index[:4]))),
                                           byte_balasan=('Total Length of Bwd Packets', 'max'))
   .sort_values('aliran', ascending=False).head(20))

## 6. Pemeriksaan ulang rancangan pada CIC-IDS2017

Transformasi log mengubah rancangan yang sebelumnya divalidasi di CIC-IDS2017 (notebook 02), jadi diuji ulang dengan
prosedur yang sama: latih pada lalu lintas normal Senin, *contamination* dipilih pada data validasi (F1 tertinggi
dengan FPR ≤ 10%), dinilai sekali pada data uji. (± 1–2 menit)

In [ ]:
F68 = json.load(open(os.path.join(CIC, 'fitur_68.json'), encoding='utf-8'))
F68 = F68['fitur'] if isinstance(F68, dict) else F68
AGG = [c + '_w300' for c in FITUR_AGREGASI]
kol = F68 + AGG + ['Label_biner', 'Skenario']
c_latih = pd.read_parquet(os.path.join(CIC, 'data', 'processed_v2', 'train_senin_normal.parquet'), columns=kol)
c_val = pd.read_parquet(os.path.join(CIC, 'data', 'processed_v2', 'val.parquet'), columns=kol)
c_uji = pd.read_parquet(os.path.join(CIC, 'data', 'processed_v2', 'test.parquet'), columns=kol)
ganti = dict(zip(AGG, FITUR_AGREGASI))
for g in (c_latih, c_val, c_uji):
    g.rename(columns=ganti, inplace=True)

from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
cic = []
for nama, tr in (('Rancangan awal (standarisasi)', None), ('Dengan transformasi log', 'log')):
    p = latih(c_latih, 0.001, 0.01, 300, transformasi=tr)
    mA, mC = SatuModel(p['A']), SatuModel(p['C'])
    sA, sC = mA.skor(c_latih), mC.skor(c_latih)
    vA, vC, yv = mA.skor(c_val), mC.skor(c_val), c_val.Label_biner.values
    best = None
    for ca in (0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05):
        for cc in (0.005, 0.01, 0.02, 0.05, 0.1):
            pr = (vA > np.quantile(sA, 1 - ca)) | (vC > np.quantile(sC, 1 - cc))
            fpr = (pr & (yv == 0)).sum() / (yv == 0).sum()
            if fpr <= 0.10 and (best is None or f1_score(yv, pr) > best[0]):
                best = (f1_score(yv, pr), ca, cc)
    _, ca, cc = best
    uA, uC, y = mA.skor(c_uji), mC.skor(c_uji), c_uji.Label_biner.values
    ta, tc = np.quantile(sA, 1 - ca), np.quantile(sC, 1 - cc)
    pr = (uA > ta) | (uC > tc)
    r = {'rancangan': nama, 'contamination A': ca, 'contamination C': cc, 'precision': precision_score(y, pr),
         'recall': recall_score(y, pr), 'F1': f1_score(y, pr), 'FPR': (pr & (y == 0)).sum() / (y == 0).sum(),
         'ROC-AUC': roc_auc_score(y, np.maximum(uA / ta, uC / tc))}
    for s, label in (('http_dos', 'DoS'), ('port_scan', 'Port scan'), ('ssh_bruteforce', 'Brute force SSH')):
        r[label] = pr[(c_uji.Skenario == s).values].mean()
    cic.append(r)
cic = pd.DataFrame(cic).set_index('rancangan')
cic.to_csv(os.path.join(HASIL, 'penalaan_cic2017.csv'))
cic.round(4)

## 7. Model final untuk sistem NIDS

Model final dilatih dengan pengaturan terpilih dari **seluruh** rekaman (latih + validasi + uji) yang sudah
dibersihkan. Hasil uji di atas disimpan di dalam model dan tampil di halaman *Model & evaluasi* pada dashboard.
Di server, model yang sama dihasilkan oleh `latih_model.py` (nilai bawaannya sudah mengikuti hasil penalaan ini).

In [ ]:
df['Kategori'] = kategori(df)
buang, tr = VARIAN[V_PILIH]
bersih = df[~df.Kategori.isin(buang)]
u = uji.iloc[:, 1]
c_log = cic.loc['Dengan transformasi log']
paket = latih(bersih, CA, CC, 300, transformasi=tr, info={
    'server_ip': sorted(SERVER_IP), 'cara_latih': 'dibersihkan (%s)' % V_PILIH,
    'periode_latih': [str(df.Waktu.min()), str(df.Waktu.max())], 'jumlah_aliran_latih': int(len(bersih)),
    'evaluasi': {'bagian': [
        {'judul': 'Uji pada data VPS perusahaan (8–11 Nov 2026)',
         'keterangan': 'Model dilatih 22 Okt–3 Nov, pengaturan dipilih pada 4–7 Nov, diuji sekali pada 8–11 Nov. Label berbasis aturan.',
         'metrik': {'Serangan intens (IP)': u['Serangan intens (IP)'], 'Pemindaian terdistribusi (IP)': u['Pemindaian terdistribusi (IP)'],
                    'Pemindai lambat (IP)': u['Pemindai lambat (IP)'], 'Alarm palsu pengunjung': '%.2f%% aliran' % u['Pengunjung berdata (%)'],
                    'Notifikasi per hari': '%.1f' % u['alarm/hari']}},
        {'judul': 'Rancangan dengan transformasi log pada CIC-IDS2017 (data uji)',
         'keterangan': 'Contamination dipilih pada data validasi CIC (F1 tertinggi, FPR ≤ 10%).',
         'metrik': {'F1-score': '%.3f' % c_log.F1, 'Precision': '%.1f%%' % (100 * c_log.precision),
                    'Recall': '%.1f%%' % (100 * c_log.recall), 'FPR': '%.1f%%' % (100 * c_log.FPR),
                    'ROC-AUC': '%.3f' % c_log['ROC-AUC'], 'DoS': '%.1f%%' % (100 * c_log.DoS),
                    'Port scan': '%.1f%%' % (100 * c_log['Port scan']), 'Brute force SSH': '%.1f%%' % (100 * c_log['Brute force SSH'])}}]}})
import joblib
MODEL = os.path.join(HASIL, 'model_vps_perusahaan_final.joblib')
joblib.dump(paket, MODEL)
print('Data latih final: %s aliran | ambang A = %.4f | ambang C = %.4f | versi %s' % (
    f'{len(bersih):,}', paket['A']['threshold'], paket['C']['threshold'], paket['versi']))
print('Model disimpan:', MODEL)